# 🔍 메모리 진단 및 시스템 모니터링

이 노트북은 현재 시스템의 메모리 상태를 종합적으로 진단하고 모니터링합니다.

## 📋 진단 항목
1. **시스템 메모리 상태**: RAM 사용량, 가용 메모리
2. **프로세스 메모리 분석**: 현재 Python 프로세스의 메모리 사용량
3. **가비지 컬렉션 진단**: Python GC 통계 및 메모리 누수 분석
4. **메모리 최적화**: 메모리 정리 및 최적화 유틸리티

In [1]:
# ==============================================
# 📦 시스템 라이브러리 임포트
# ==============================================

import psutil
import gc
import sys
import os
import time
import tracemalloc
import platform
from datetime import datetime

print("🔧 메모리 진단 시스템 초기화")
print("=" * 50)
print(f"🖥️  운영체제: {platform.system()} {platform.release()}")
print(f"🐍 Python 버전: {sys.version}")
print(f"📊 psutil 버전: {psutil.__version__}")
print(f"⏰ 진단 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)
print("✅ 라이브러리 로드 완료!")

🔧 메모리 진단 시스템 초기화
🖥️  운영체제: Windows 10
🐍 Python 버전: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 12:58:53) [MSC v.1929 64 bit (AMD64)]
📊 psutil 버전: 5.9.0
⏰ 진단 시작: 2025-07-25 18:07:23
✅ 라이브러리 로드 완료!


In [3]:
# ==============================================
# 📊 메모리 사용량 분석 함수
# ==============================================

def get_memory_info():
    """시스템 메모리 정보 조회"""
    memory = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    print("🖥️  시스템 메모리 상태")
    print("=" * 40)
    print(f"전체 메모리:     {memory.total / (1024**3):.2f} GB")
    print(f"사용 중 메모리:   {memory.used / (1024**3):.2f} GB")
    print(f"가용 메모리:     {memory.available / (1024**3):.2f} GB")
    print(f"메모리 사용률:   {memory.percent:.1f}%")
    # Windows에서는 buffers 속성이 없으므로 제거
    print()
    print("💾 스왑 메모리 상태")
    print("=" * 40)
    print(f"전체 스왑:      {swap.total / (1024**3):.2f} GB")
    print(f"사용 중 스왑:    {swap.used / (1024**3):.2f} GB")
    print(f"스왑 사용률:    {swap.percent:.1f}%")
    
    return memory, swap

def get_process_memory():
    """현재 프로세스 메모리 정보"""
    process = psutil.Process()
    memory_info = process.memory_info()
    memory_percent = process.memory_percent()
    
    print("🐍 Python 프로세스 메모리")
    print("=" * 40)
    print(f"RSS (물리 메모리):  {memory_info.rss / (1024**2):.1f} MB")
    print(f"VMS (가상 메모리):  {memory_info.vms / (1024**2):.1f} MB")
    print(f"프로세스 메모리율:  {memory_percent:.2f}%")
    
    # 추가 메모리 정보 (Windows에서 사용 가능한 경우)
    try:
        memory_full = process.memory_full_info()
        print(f"USS (고유 메모리):  {memory_full.uss / (1024**2):.1f} MB")
        print(f"PSS (공유 메모리):  {memory_full.pss / (1024**2):.1f} MB")
    except AttributeError:
        print("상세 메모리 정보는 이 플랫폼에서 지원되지 않습니다.")
    
    return memory_info, memory_percent

# 즉시 실행
print("🔍 현재 메모리 상태 분석")
print("=" * 50)
memory, swap = get_memory_info()
print()
memory_info, memory_percent = get_process_memory()

🔍 현재 메모리 상태 분석
🖥️  시스템 메모리 상태
전체 메모리:     7.73 GB
사용 중 메모리:   6.65 GB
가용 메모리:     1.08 GB
메모리 사용률:   86.0%

💾 스왑 메모리 상태
전체 스왑:      7.73 GB
사용 중 스왑:    2.49 GB
스왑 사용률:    32.3%

🐍 Python 프로세스 메모리
RSS (물리 메모리):  86.3 MB
VMS (가상 메모리):  87.8 MB
프로세스 메모리율:  1.09%
USS (고유 메모리):  69.4 MB
상세 메모리 정보는 이 플랫폼에서 지원되지 않습니다.


In [4]:
# ==============================================
# 📈 실시간 프로세스 메모리 모니터링
# ==============================================

def monitor_memory_usage(duration=30, interval=5):
    """지정된 시간 동안 메모리 사용량 모니터링"""
    print(f"📊 {duration}초 동안 {interval}초 간격으로 모니터링 시작...")
    print("=" * 60)
    print(f"{'시간':<12} {'RSS(MB)':<10} {'VMS(MB)':<10} {'CPU%':<8} {'메모리%':<8}")
    print("-" * 60)
    
    process = psutil.Process()
    start_time = time.time()
    
    while time.time() - start_time < duration:
        current_time = datetime.now().strftime('%H:%M:%S')
        memory_info = process.memory_info()
        cpu_percent = process.cpu_percent()
        memory_percent = process.memory_percent()
        
        print(f"{current_time:<12} {memory_info.rss/(1024**2):<10.1f} "
              f"{memory_info.vms/(1024**2):<10.1f} {cpu_percent:<8.1f} {memory_percent:<8.2f}")
        
        time.sleep(interval)

def check_memory_pressure():
    """메모리 압박 상태 확인"""
    memory = psutil.virtual_memory()
    
    print("⚠️  메모리 압박 상태 분석")
    print("=" * 40)
    
    if memory.percent > 90:
        print("🚨 위험: 메모리 사용률이 90% 초과")
        status = "CRITICAL"
    elif memory.percent > 80:
        print("⚠️  경고: 메모리 사용률이 80% 초과")
        status = "WARNING"
    elif memory.percent > 70:
        print("📢 주의: 메모리 사용률이 70% 초과")
        status = "CAUTION"
    else:
        print("✅ 정상: 메모리 사용률 안정")
        status = "NORMAL"
    
    # 가용 메모리 확인
    available_gb = memory.available / (1024**3)
    if available_gb < 0.5:
        print("🚨 가용 메모리가 500MB 미만입니다!")
    elif available_gb < 1.0:
        print("⚠️  가용 메모리가 1GB 미만입니다.")
    
    return status

# 현재 메모리 압박 상태 확인
memory_status = check_memory_pressure()
print()
print(f"현재 메모리 상태: {memory_status}")

⚠️  메모리 압박 상태 분석
⚠️  경고: 메모리 사용률이 80% 초과

현재 메모리 상태: WARNING


In [5]:
# ==============================================
# 🗑️  가비지 컬렉션 진단
# ==============================================

def analyze_garbage_collection():
    """가비지 컬렉션 통계 분석"""
    print("🗑️  가비지 컬렉션 상태")
    print("=" * 40)
    
    # GC 통계
    gc_stats = gc.get_stats()
    for i, stat in enumerate(gc_stats):
        print(f"세대 {i}: 컬렉션 {stat['collections']}회, "
              f"수집 {stat['collected']}개, 미수집 {stat['uncollectable']}개")
    
    # 현재 객체 수
    print(f"\n현재 추적 중인 객체 수: {len(gc.get_objects())}")
    
    # 순환 참조 확인
    print(f"가비지 컬렉션 활성화: {gc.isenabled()}")
    print(f"GC 임계값: {gc.get_threshold()}")
    
    return gc_stats

def force_garbage_collection():
    """강제 가비지 컬렉션 실행"""
    print("🔧 강제 가비지 컬렉션 실행 중...")
    
    # 수집 전 메모리
    process = psutil.Process()
    before_memory = process.memory_info().rss / (1024**2)
    
    # 가비지 컬렉션 실행
    collected = gc.collect()
    
    # 수집 후 메모리
    after_memory = process.memory_info().rss / (1024**2)
    freed_memory = before_memory - after_memory
    
    print(f"수집된 객체: {collected}개")
    print(f"메모리 변화: {before_memory:.1f}MB → {after_memory:.1f}MB")
    print(f"해제된 메모리: {freed_memory:.1f}MB")
    
    return collected, freed_memory

def find_large_objects(limit=10):
    """큰 객체들 찾기"""
    print(f"📋 가장 큰 객체 {limit}개")
    print("=" * 40)
    
    # 모든 객체 수집
    objects = gc.get_objects()
    
    # 크기별 정렬
    try:
        large_objects = []
        for obj in objects:
            try:
                size = sys.getsizeof(obj)
                obj_type = type(obj).__name__
                large_objects.append((size, obj_type, str(obj)[:50]))
            except:
                continue
        
        large_objects.sort(reverse=True)
        
        for i, (size, obj_type, obj_str) in enumerate(large_objects[:limit]):
            print(f"{i+1:2d}. {size:>10,} bytes - {obj_type:<15} - {obj_str}")
    
    except Exception as e:
        print(f"객체 분석 중 오류: {e}")

# GC 분석 실행
print("🔍 가비지 컬렉션 진단 시작")
print("=" * 50)
gc_stats = analyze_garbage_collection()
print()
collected, freed = force_garbage_collection()
print()
find_large_objects()

🔍 가비지 컬렉션 진단 시작
🗑️  가비지 컬렉션 상태
세대 0: 컬렉션 268회, 수집 1144개, 미수집 0개
세대 1: 컬렉션 24회, 수집 1206개, 미수집 0개
세대 2: 컬렉션 2회, 수집 3개, 미수집 0개

현재 추적 중인 객체 수: 124437
가비지 컬렉션 활성화: True
GC 임계값: (700, 10, 10)

🔧 강제 가비지 컬렉션 실행 중...
수집된 객체: 1068개
메모리 변화: 88.4MB → 88.4MB
해제된 메모리: -0.0MB

📋 가장 큰 객체 10개
 1.    153,752 bytes - list            - [Token(type=3, string='"""Main IPython class."""',
 2.    153,752 bytes - list            - [0, 25, 26, 27, 105, 106, 160, 161, 225, 226, 282,
 3.    147,552 bytes - defaultdict     - defaultdict(<class 'list'>, {1: [<ast.Expr object 
 4.     73,808 bytes - dict            - {1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
 5.     51,968 bytes - dict            - {'sys': <module 'sys' (built-in)>, 'builtins': <mo
 6.     36,952 bytes - dict            - {140722013576592: <weakref at 0x0000028D3C194C20; 
 7.     33,048 bytes - list            - [0, 26, 27, 106, 161, 226, 283, 285, 360, 419, 498
 8.     33,048 bytes - list            - ['"""Main IPython class."""\n', '\n',

In [6]:
# ==============================================
# 🛠️  메모리 최적화 유틸리티
# ==============================================

def cleanup_variables():
    """불필요한 변수들 정리"""
    print("🧹 변수 정리 시작...")
    
    # 현재 글로벌 변수 확인
    global_vars = list(globals().keys())
    print(f"현재 글로벌 변수 개수: {len(global_vars)}")
    
    # 큰 변수들 찾기
    large_vars = []
    for var_name in global_vars:
        if not var_name.startswith('_'):  # 내부 변수 제외
            try:
                var_obj = globals()[var_name]
                size = sys.getsizeof(var_obj)
                if size > 1024:  # 1KB 이상
                    large_vars.append((var_name, size, type(var_obj).__name__))
            except:
                continue
    
    large_vars.sort(key=lambda x: x[1], reverse=True)
    
    print("큰 변수들 (1KB 이상):")
    for var_name, size, var_type in large_vars[:10]:
        print(f"  {var_name:<20}: {size:>10,} bytes ({var_type})")
    
    return large_vars

def memory_optimization_tips():
    """메모리 최적화 팁 제공"""
    print("💡 메모리 최적화 팁")
    print("=" * 40)
    tips = [
        "1. 사용하지 않는 큰 변수들을 del로 삭제하세요",
        "2. 정기적으로 gc.collect()를 호출하세요", 
        "3. 큰 데이터프레임은 청크 단위로 처리하세요",
        "4. NumPy 배열의 dtype을 최적화하세요 (float32 vs float64)",
        "5. 메모리 매핑을 사용하여 큰 파일을 처리하세요",
        "6. 판다스에서 category 타입을 활용하세요",
        "7. 불필요한 복사본 생성을 피하세요 (copy=False 사용)",
        "8. with 문을 사용하여 리소스를 자동으로 해제하세요"
    ]
    
    for tip in tips:
        print(tip)

def emergency_memory_cleanup():
    """긴급 메모리 정리"""
    print("🚨 긴급 메모리 정리 실행")
    print("=" * 40)
    
    # 메모리 정리 전 상태
    process = psutil.Process()
    before_memory = process.memory_info().rss / (1024**2)
    
    # 1. 가비지 컬렉션
    collected = gc.collect()
    
    # 2. 메모리 정리 후 상태
    after_memory = process.memory_info().rss / (1024**2)
    freed = before_memory - after_memory
    
    print(f"가비지 컬렉션으로 수집된 객체: {collected}개")
    print(f"메모리 사용량: {before_memory:.1f}MB → {after_memory:.1f}MB")
    print(f"해제된 메모리: {freed:.1f}MB")
    
    # 3. 시스템 메모리 상태 재확인
    memory = psutil.virtual_memory()
    print(f"현재 시스템 메모리 사용률: {memory.percent:.1f}%")
    print(f"가용 메모리: {memory.available / (1024**3):.2f}GB")
    
    return freed

# 메모리 최적화 도구 실행
print("🛠️  메모리 최적화 도구")
print("=" * 50)
large_vars = cleanup_variables()
print()
memory_optimization_tips()
print()
freed_memory = emergency_memory_cleanup()

🛠️  메모리 최적화 도구
🧹 변수 정리 시작...
현재 글로벌 변수 개수: 55
큰 변수들 (1KB 이상):

💡 메모리 최적화 팁
1. 사용하지 않는 큰 변수들을 del로 삭제하세요
2. 정기적으로 gc.collect()를 호출하세요
3. 큰 데이터프레임은 청크 단위로 처리하세요
4. NumPy 배열의 dtype을 최적화하세요 (float32 vs float64)
5. 메모리 매핑을 사용하여 큰 파일을 처리하세요
6. 판다스에서 category 타입을 활용하세요
7. 불필요한 복사본 생성을 피하세요 (copy=False 사용)
8. with 문을 사용하여 리소스를 자동으로 해제하세요

🚨 긴급 메모리 정리 실행
가비지 컬렉션으로 수집된 객체: 0개
메모리 사용량: 104.5MB → 103.5MB
해제된 메모리: 1.0MB
현재 시스템 메모리 사용률: 85.8%
가용 메모리: 1.10GB


In [7]:
# ==============================================
# 📋 최종 진단 요약 및 권장사항
# ==============================================

def final_diagnosis():
    """최종 진단 요약"""
    print("📋 메모리 진단 최종 요약")
    print("=" * 50)
    
    # 현재 상태 재확인
    memory = psutil.virtual_memory()
    process = psutil.Process()
    process_memory = process.memory_info()
    
    print("🖥️  시스템 상태:")
    print(f"   전체 메모리: {memory.total / (1024**3):.2f} GB")
    print(f"   사용 중: {memory.used / (1024**3):.2f} GB ({memory.percent:.1f}%)")
    print(f"   가용 메모리: {memory.available / (1024**3):.2f} GB")
    
    print(f"\n🐍 Python 프로세스:")
    print(f"   RSS 메모리: {process_memory.rss / (1024**2):.1f} MB")
    print(f"   VMS 메모리: {process_memory.vms / (1024**2):.1f} MB")
    print(f"   프로세스 메모리율: {process.memory_percent():.2f}%")
    
    # 권장사항
    print(f"\n💡 권장사항:")
    
    if memory.percent > 85:
        print("   🚨 시스템 메모리 사용률이 높습니다!")
        print("   - 불필요한 프로그램을 종료하세요")
        print("   - 메모리 집약적인 작업을 줄이세요")
        
    if memory.available < 1024**3:  # 1GB 미만
        print("   ⚠️  가용 메모리가 부족합니다!")
        print("   - 대용량 데이터 처리를 청크 단위로 진행하세요")
        print("   - 변수를 적극적으로 삭제하세요 (del variable)")
        
    if process_memory.rss > 500 * 1024**2:  # 500MB 초과
        print("   📢 Python 프로세스 메모리 사용량이 높습니다")
        print("   - gc.collect()를 정기적으로 실행하세요")
        print("   - 큰 객체들을 확인하고 정리하세요")
    
    print(f"\n⏰ 진단 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 50)

# 최종 진단 실행
final_diagnosis()

print("\n🎯 즉시 실행 가능한 메모리 정리 명령어:")
print("gc.collect()  # 가비지 컬렉션 실행")
print("del variable_name  # 특정 변수 삭제") 
print("globals().clear()  # 모든 글로벌 변수 삭제 (주의!)")
print("\n✅ 메모리 진단이 완료되었습니다!")

📋 메모리 진단 최종 요약
🖥️  시스템 상태:
   전체 메모리: 7.73 GB
   사용 중: 6.63 GB (85.8%)
   가용 메모리: 1.10 GB

🐍 Python 프로세스:
   RSS 메모리: 103.5 MB
   VMS 메모리: 104.2 MB
   프로세스 메모리율: 1.31%

💡 권장사항:
   🚨 시스템 메모리 사용률이 높습니다!
   - 불필요한 프로그램을 종료하세요
   - 메모리 집약적인 작업을 줄이세요

⏰ 진단 완료: 2025-07-25 18:08:26

🎯 즉시 실행 가능한 메모리 정리 명령어:
gc.collect()  # 가비지 컬렉션 실행
del variable_name  # 특정 변수 삭제
globals().clear()  # 모든 글로벌 변수 삭제 (주의!)

✅ 메모리 진단이 완료되었습니다!
